# The Tool-Using Agent: A Workflow Pattern for Bounded, Efficient and Reproducible Agent Systems

Tutorial by [Lilly Thomas](https://github.com/lillythomas).

This notebook demonstrates a workflow pattern and core architectural considerations for building LLM agents that are:

1. **Trustworthy** - Human-in-the-loop gates, deterministic confirmation handling, templated responses
2. **Reproducible** - Structured schemas, explicit state management, status-based flow control
3. **Efficient** - State-based parameter passing to reduce token usage and prevent hallucination

We'll build a minimal **Snow Cover Analysis Agent** that queries [MODIS Snow Cover data](https://nsidc.org/data/mod10a1/versions/61#anchor-documentation) from the [Microsoft Planetary Computer STAC API](https://planetarycomputer.microsoft.com/).

---

### When to Use This Pattern

The **tool-using workflow pattern** demonstrated here is ideal for **bounded, well-defined use cases** where:

- You know the types of questions and analyses users will request
- The workflow can be decomposed into a finite set of discrete tools
- Predictability and reproducibility matter more than flexibility

**Why it's resource-efficient:**

Autonomous agent architectures spend tokens on open-ended research, planning, and self-reflection. This pattern avoids that overhead by encoding the workflow directly into tools.

| Aspect | Autonomous Agents | Tool-Using Workflow |
|--------|-------------------|----------------------|
| **Planning overhead** | High (multi-step reasoning) | Low (tools encode the workflow) |
| **Token usage** | Variable, often high | Predictable, minimal |
| **Latency** | Multiple LLM round-trips | Fewer, faster cycles |
| **Behavior** | Emergent, harder to debug | Deterministic, auditable |

**Trade-offs to consider:**

- **Bounded capability**: The agent can only do what its tools allow. Novel requests outside the tool set may be declined, redirected to supported capabilities, handled poorly, or cause the agent to hallucinate.
- **Upfront design cost**: You must anticipate user needs and encode them as tools. This requires domain expertise.
- **Less adaptive**: Unlike agents that can research and improvise, this pattern won't discover new approaches on its own.

**When to use something else:**

If your use case requires open-ended exploration, code generation, or handling truly novel requests, consider:
- **Autonomous Agents and/or RPI** for exploratory tasks
- **RAG + reasoning** for knowledge-intensive Q&A

This notebook focuses on the **constrained, production-ready** end of the spectrum—where reliability is prioritized over generality.

---

## Part 1: The Evolution of Tool-Using LLM Agents

### From ReAct to Native Tool Use

**ReAct (2022)** was a groundbreaking prompting technique that enabled LLMs to use tools *before* models had native tool-calling capabilities. It worked by having the LLM generate structured text that was then parsed:

```
[Human]: What was the snow cover in Seattle last January?

Thought: I need to find the user's location first
Action: get_location
Action Input: {"query": "Seattle"}
Observation: {"place": "Seattle, WA", "bbox": [...]}
Thought: Location found. Now I need to set the date range.
...
```

This was clever but fragile—if the LLM didn't follow the exact format, parsing would fail.

**Native Tool Use ([2023+](https://openai.com/index/function-calling-and-other-api-updates/))** solved this by building tool calling directly into the model provider's API. Instead of parsing text, the model returns structured JSON that's guaranteed to be valid:

```json
{
  "tool_calls": [
    {"name": "get_location", "arguments": {"query": "Seattle"}},
    {"name": "set_date_range", "arguments": {"start": "2026-01-01", "end": "2026-01-31"}}
  ]
}
```

### Why This Matters

| Aspect | ReAct (Text Parsing) | Native Tool Use |
|--------|---------------------|-----------------|
| **Reliability** | Can fail if format is wrong | API-enforced, always valid |
| **Parallel calls** | Difficult (sequential by nature) | Native support |
| **Token efficiency** | Higher (includes "Thought:" text) | Lower (just function calls) |
| **Model support** | Any LLM (via prompting) | Requires API support |

**Today, native tool use is the standard** for production agents. ReAct's contribution was proving that LLMs *could* effectively use tools—the industry then built that capability directly into the models.

### Terminology

Different providers use different names for the same capability:

| Provider | Term |
|----------|------|
| OpenAI | "Function Calling" → now "Tool Use" |
| Anthropic | "Tool Use" |
| Google | "Function Calling" |
| Academic | "Tool-Augmented LLMs" |

**This notebook uses "Tool Use" as the provider-agnostic term.**

### References

- **ReAct Paper**: [ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629) (Yao et al., 2022)
- **OpenAI Tool Use**: [Function Calling Guide](https://platform.openai.com/docs/guides/function-calling)
- **Anthropic Tool Use**: [Tool Use Documentation](https://docs.anthropic.com/en/docs/build-with-claude/tool-use/overview)
- **LangGraph**: [Documentation](https://langchain-ai.github.io/langgraph/)

---

## Part 2: Architectural Considerations for Trustworthy Tool-Using Agents

Regardless of which LLM or framework you use, these architectural considerations make your agent more trustworthy, reproducible, and efficient.

| Principle | Implementation | Benefit |
|-----------|----------------|--------|
| **LLM as Navigator, Not Data Carrier** | Tools read large data from state, not LLM parameters | Prevents hallucination, reduces tokens |
| **Human-Controlled Decisions** | Critical choices require explicit user confirmation | Trustworthiness, auditability |
| **Deterministic Confirmation** | Simple yes/no bypasses LLM entirely | Reduces tokens, faster responses |
| **Templated Responses** | All user-facing messages are predefined | Consistency, no rephrasing |
| **Structured Tool Outputs** | Pydantic schemas with status codes | Deterministic flow control |

### Why Deterministic Confirmation?

When a user responds with a simple "yes" or "no" to a confirmation prompt, there's no need to spend tokens having the LLM interpret it. We can:

1. **Classify the response deterministically** (term matching)
2. **Handle simple responses without LLM** (save tokens, reduce latency)
3. **Only route complex responses to the LLM** (e.g., "no, I meant East River, CO instead")

This is mainly an **efficiency optimization** - it reduces token usage and response time for the common case of simple confirmations.

---

## Part 3: Setup

Set your API key:
- **Anthropic**: `export ANTHROPIC_API_KEY="your-key"` - Claude access

**Data Sources:**
- **Geocoding**: [Nominatim/OpenStreetMap](https://nominatim.org/) (free, no API key required)
- **Snow Cover Data**: [MODIS Snow Cover Daily via Planetary Computer STAC](https://planetarycomputer.microsoft.com/dataset/modis-10A1-061) (free, no API key required)

---

### Using Jupyter AI Chat

This project includes [Jupyter AI](https://jupyter-ai.readthedocs.io/), an AI assistant built into JupyterLab.

To open the chat panel, click **+ New chat** or look for the chat icon in the left sidebar.

**To use an AI persona, install one first:**

```bash
# Option 1: Install OpenCode (free models included)
curl -fsSL https://opencode.ai/install | bash

# Option 2: Install Claude adapter (requires Anthropic API key)
curl -fsSL https://claude.ai/install.sh | bash
```
After installing, restart JupyterLab. Then in the chat, installed personas should show up in the lower drop down menu. You can begin chatting with whichever you select. You will have to [authenticate if using Claude](https://code.claude.com/docs/en/quickstart).

In [ ]:
# Install required dependencies in the active kernel
# If you don't have uv, use: %pip install -e .
!uv pip install -e .

In [ ]:
# If neither of those work, use this to install runtime dependencies
# %pip install "langgraph>=0.2.0" "langchain-anthropic>=0.3.0" "langchain-core>=0.3.0" "numpy>=1.26.0" "requests>=2.31.0" "rioxarray>=0.15.0" "planetary-computer>=0.5.0" "pandas>=2.0.0"

In [ ]:
import os

# Set your Anthropic API key
os.environ["ANTHROPIC_API_KEY"] = "your-API-key"

from tool_using_snow_agent import (
    init_state,
    build_tool_calling_agent,
    get_location,
    set_date_range,
    run_analysis,
    get_response,
    classify_response,
    handle_confirmation,
    PRICING,
    analyze_token_usage,
    get_state,
)
from langchain_core.messages import HumanMessage

print('Imports complete. Package loaded.')

**LLM configuration** is handled in `tool_using_snow_agent/llm.py`. It creates a `ChatAnthropic` instance with prompt caching enabled.

---

## Part 4: Implementing the Architecture Recommendations

Now let's implement each recommendation with working code.

### Recommendation 1: Structured Response System (Templated Responses)

**Implementation:** `tool_using_snow_agent/responses.py` defines `ResponseStatus`, `RESPONSES`, `get_response()`, and `build_tool_response()`.

In [ ]:
from tool_using_snow_agent.responses import ResponseStatus, get_response, build_tool_response

# Example: get a templated response
print(get_response('location_resolved', place='Mesa County, CO'))  # Grand Mesa, CO

# Example: build a standardized tool response
print(build_tool_response(ResponseStatus.COMPLETE, 'Date range set.'))

### Recommendation 2: State-Based Parameter Passing (LLM as Navigator)

**Implementation:** `tool_using_snow_agent/state.py` manages the shared `_state` dictionary and provides `init_state()` and `get_state()`. 

Tip: For production, move domain state [into the graph](https://reference.langchain.com/python/langgraph.prebuilt/tool_node/InjectedState) and enable [checkpointing](https://docs.langchain.com/oss/python/langgraph/checkpointers) for thread safety and memory persistence. 

In [ ]:
from tool_using_snow_agent.state import init_state, get_state

init_state()
print('State keys:', list(get_state().keys()))

### Recommendation 3: Deterministic Confirmation Handling

**Purpose:** Reduce token usage by handling simple yes/no responses without calling the LLM. Only complex responses (corrections, clarifications) go to the LLM.

**Implementation:** `tool_using_snow_agent/confirmation.py` defines `ConfirmationType`, `classify_response()`, and `handle_confirmation()`.

In [ ]:
from tool_using_snow_agent.confirmation import classify_response, handle_confirmation

# Deterministic classification
for response in ['yes', 'Yeah!', 'no', 'no, I meant East River, CO']:
    cls = classify_response(response)
    llm_needed = 'LLM needed' if cls == 'complex' else 'No LLM (tokens saved!)'
    print(f"  '{response}' -> {cls} -> {llm_needed}")

### Recommendation 4: Tools with State Management

`tool_using_snow_agent/services.py` contains `geocode_location()` and `fetch_snow_data()` for Nominatim geocoding and MODIS data retrieval.

In [ ]:
from tool_using_snow_agent.services import geocode_location

# Geocode a test location
result = geocode_location('Grand Mesa, CO') # 'Mesa County, CO'
if result:
    print(result['name'][:60])
    print('bbox:', result['bbox'])
    print('Geometry type returned: ', result['geometry']['type'])

Each tool:
- Reads/writes to shared state (not LLM parameters)
- Returns structured responses with status codes
- Uses templated messages for consistency

`tool_using_snow_agent/tools.py` wraps `get_location`, `set_date_range`, and `run_analysis` as LangChain `@tool` functions.

In [ ]:
from tool_using_snow_agent.tools import get_location, set_date_range, run_analysis

print('Tools:', [t.name for t in [get_location, set_date_range, run_analysis]])

---

## Part 5: Building and Invoking the Agent

`tool_using_snow_agent/agent.py` defines `SYSTEM_PROMPT` and `build_tool_calling_agent()` which builds the LangGraph workflow.

Start the agent with a sample query.  

`agent.invoke()` passes the initial `HumanMessage` through the LangGraph workflow. After the agent finishes, we iterate over `result['messages']`, label each message by its role (`Human`, `AI`, or `Tool`), and print a preview of its content.

In [ ]:
from tool_using_snow_agent.agent import build_tool_calling_agent, SYSTEM_PROMPT

agent = build_tool_calling_agent()
print('Agent built:', type(agent).__name__)

In [ ]:
# Initialize fresh state and run the agent
init_state()
agent = build_tool_calling_agent()

result = agent.invoke({
    "messages": [HumanMessage(content="Analyze snow cover in Mesa County, Colorado for winter 2023")] # NOTE: the last item in the collection is from June, 2025 despite the metadata implying ongoing
})

for msg in result['messages']:
    role = msg.__class__.__name__.replace('Message', '')
    content = getattr(msg, 'content', '')
    if content:
        print(f'[{role}]: {content[:1000]}{'...' if len(str(content)) > 1000 else ''}')

---

## Part 6: Continuing the Conversation

The agent returned `pending_confirmation` status, which means it's waiting for user input before proceeding. Let's continue the conversation.

In [ ]:
# OPTIONAL CELL: Continue the conversation (adjust the location to see the non-determinstic response pathway)
result = agent.invoke({
    'messages': result['messages'] + [HumanMessage(content='no, I meant East River, CO')]
})

for msg in result['messages']:
    role = msg.__class__.__name__.replace('Message', '')
    content = getattr(msg, 'content', '')
    if content:
        print(f'[{role}]: {content[:1000]}{'...' if len(str(content)) > 1000 else ''}')

In [ ]:
# Required cell: Continue the conversation (confirm the location)
result = agent.invoke({
    'messages': result['messages'] + [HumanMessage(content='yes')]
})

for msg in result['messages']:
    role = msg.__class__.__name__.replace('Message', '')
    content = getattr(msg, 'content', '')
    if content:
        print(f'[{role}]: {content[:1000]}{'...' if len(str(content)) > 1000 else ''}')

---

## LLM Call Breakdown

This section shows what caused each LLM API call in the agent run. Understanding the trigger for each call makes the flow auditable and helps identify where token costs come from.

In [ ]:
from tool_using_snow_agent import explain_llm_calls

explain_llm_calls(result["messages"])

---

## Part 7: Token Usage & Cost Analysis

This section analyzes the actual LLM API calls made during the agent run, including token usage and costs.

### Understanding Prompt Caching

Anthropic's [prompt caching](https://platform.claude.com/docs/en/build-with-claude/prompt-caching) reduces costs by caching repeated content (system prompts, tool definitions) across API calls.

| Token Type | Description | [Pricing (Sonnet 4.5)](https://platform.claude.com/docs/en/about-claude/pricing) |
|------------|-------------|---------------------|
| **Base Input** | Uncached input tokens | 3.00 USD per 1M tokens |
| **Cache Write** | Tokens written to cache (first use) | 3.75 USD per 1M tokens (25 percent premium) |
| **Cache Read** | Tokens read from cache (subsequent uses) | 0.30 USD per 1M tokens (90 percent discount!) |
| **Output** | Generated tokens | 15.00 USD per 1M tokens |


### How Caching Works in This Agent

1. **First API call**: The system prompt and tool definitions are written to the cache (`ephemeral_5m_input_tokens`)
2. **Subsequent calls**: The same content is read from cache (`cache_read`), saving ~90% on those tokens
3. **Cache TTL**: 5 minutes for ephemeral cache; if you re-run within 5 minutes, even the first call benefits from the previous session's cache

### Interpreting the Results

- **Cache Read > 0 on Call #1**: The cache is "warm" from a previous run (within 5 minutes)
- **Cache Read = 0 on Call #1**: "Cold start" - first run or cache expired
- **High Cache Read ratio**: Good! Most input tokens are being read from cache at 90% discount

`tool_using_snow_agent/pricing.py` defines `PRICING` and `analyze_token_usage()` for cost analysis.

In [ ]:
from tool_using_snow_agent.pricing import PRICING, analyze_token_usage

print('Claude Sonnet 4.5 Pricing:')
print(f"  Base input:   ${PRICING['input']}/1M tokens")
print(f"  Cache write:  ${PRICING['cache_write']}/1M tokens")
print(f"  Cache read:   ${PRICING['cache_read']}/1M tokens")
print(f"  Output:       ${PRICING['output']}/1M tokens")
print()

df = analyze_token_usage(result['messages'])
df

---

## Part 8: Summary

### The Evolution of Tool-Using Agents

| Era | Approach | How It Works |
|-----|----------|--------------|
| **2022** | ReAct | LLM generates "Thought/Action" text, parsed by framework |
| **2023+** | Native Tool Use | LLM returns structured JSON via API, no parsing needed |

**Native tool use is now the standard** for production agents. It's more reliable, efficient, and supports parallel tool calls.

### Architectural Considerations for Trustworthy Agents

| Recommendation | Purpose | Implementation |
|---------|---------|----------------|
| **State-Based Parameters** | Prevent hallucination, reduce tokens | Tools read large data from state, not LLM |
| **Templated Responses** | Consistent, auditable messages | All user-facing text predefined |
| **Status Codes** | Deterministic flow control | `complete`, `pending_confirmation`, `error` |
| **Deterministic Confirmation** | Reduce tokens for simple yes/no | Bypass LLM for simple responses |
| **Human-in-the-Loop Gates** | Critical decisions need approval | `pending_confirmation` stops agent |

### Key Takeaway

> **The LLM should be a decision-maker, not a data carrier.**
> 
> - Large data flows through state, not message history
> - Critical decisions require human confirmation
> - All user-facing messages are templated
> - Simple yes/no responses can bypass the LLM to save tokens

### References

- [ReAct Paper](https://arxiv.org/abs/2210.03629) - Yao et al., 2022 (historical context)
- [OpenAI Tool Use](https://platform.openai.com/docs/guides/function-calling)
- [Anthropic Tool Use](https://docs.anthropic.com/en/docs/build-with-claude/tool-use/overview)
- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/)